In [1]:
import polars as pl
import pandas as pd
import gc
from features import apply_type_casting, generate_features, cat_features
from catboost import CatBoostClassifier

In [ ]:
df_train = pl.scan_parquet('../data/train_full.parquet').drop('target')
df_pretest = apply_type_casting(pl.scan_parquet('../data/pretest.parquet'))
df_test = apply_type_casting(pl.scan_parquet('../data/test.parquet'))

df_all = pl.concat([df_train, df_pretest, df_test])
df_all = df_all.unique(subset='event_id', keep='last')
df_all = generate_features(df_all)

df_test_features = df_all.join(
    df_test.select('event_id'),
    on='event_id'
)

del df_train, df_pretest, df_test, df_all
gc.collect()

df_test_features_collected = df_test_features.collect().to_pandas()
print(f'Размер датасета: {df_test_features_collected.shape}')

Размер датасета: (633683, 42)


In [ ]:
event_id = df_test_features_collected['event_id']
X_test = df_test_features_collected.drop(['customer_id', 'event_id', 'event_dttm', 'event_date'], axis=1)


model = CatBoostClassifier()
model.load_model('../models/catboost_v4.cbm')
preds = model.predict_proba(X_test)[:, 1]

df_submit = pd.DataFrame({
    'event_id': event_id,
    'predict': preds
})

df_submit.to_csv('../data/submission_v4.csv', index=False)